In [ ]:
import torch
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import time
from gdv_utils import calculate_gdv 

# 1. SETUP DEVICE PER LA GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo per analisi GDV: {device}\n")

# 2. MAPPING DEI DATASET COMPLETO (Nome -> UCI ID)
UCI_DATASETS = {
    "Autism Screening Adult": 426,
    "Bank Marketing": 222,
    "Blood Transfusion": 176,
    "Breast Cancer": 14,
    "Breast Cancer Coimbra": 451,
    "Breast Cancer Wisconsin (Diag.)": 17,
    "Cardiotocography": 193,
    "Chess (King-Rook vs. Pawn)": 22,
    "Cirrhosis Patient Survival": 878,
    "Congressional Voting": 105,
    "Contraceptive Method Choice": 30,
    "Credit Approval": 27,
    "Diff. Thyroid Cancer Recurrence": 848,
    "Drug Consumption": 373,
    "Haberman's Survival": 43,
    "Hayes-Roth": 44,
    "Heart Disease": 45,
    "Hepatitis C Virus (HCV)": 571,
    "ILPD (Indian Liver Patient)": 225,
    "ISOLET": 54,
    "Image Segmentation": 50,
    "Ionosphere": 52,
    "Letter Recognition": 59,
    "Mammographic Mass": 161,
    "Molecular Biology (Splice)": 69,
    "Musk (Version 2)": 75,
    "National Poll on Healthy Aging": 936,
    "NHANES Age Prediction": 887,
    "Optical / Pen-Based Digits": 80,
    "Page Blocks Classification": 78,
    "Polish Companies Bankruptcy": 365,
    "Predict Students' Dropout": 697,
    "Raisin": 850,
    "Room Occupancy Estimation": 864,
    "Spambase": 94,
    "SPECTF Heart": 96,
    "Statlog (Australian)": 143,
    "Statlog (German)": 144,
    "Statlog (Vehicle Silhouettes)": 149,
    "Steel Plates Faults": 198,
    "Student Performance": 320,
    "Taiwanese Bankruptcy": 572,
    "Vertebral Column": 212,
    "Waveform Database Gen.": 107,
    "Website Phishing": 379,
    "Yeast": 110
}

def clean_and_prepare_data(X, y, max_samples=15000):
    """
    Pipeline di pulizia universale per i dataset UCI.
    Include limite di campioni per evitare OOM.
    I dataset Multi-Target andranno in errore (IndexError) e verranno esclusi.
    """
    # 1. Standardizzazione formati in Pandas DataFrame
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    if not isinstance(y, (pd.DataFrame, pd.Series)):
        y = pd.Series(y.ravel() if hasattr(y, 'ravel') else y)

    # 2. Rimozione dei NaN sui target
    valid_idx = y.dropna().index
    X = X.loc[valid_idx].copy()
    y = y.loc[valid_idx].copy()

    # Limite di 15000: sufficiente per stabilità statistica GDV, sicuro per la RAM
    if len(X) > max_samples:
        sampled_idx = X.sample(n=max_samples, random_state=42).index
        X = X.loc[sampled_idx]
        y = y.loc[sampled_idx]

    # 3. Encoding categoriali
    X = pd.get_dummies(X, drop_first=True)

    # 4. Imputazione e Scaling
    imputer = SimpleImputer(strategy='mean')
    X_imputed = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    # 5. Encoding Labels
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y.values.ravel())

    # 6. Spostamento su device (GPU/CPU)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_encoded, dtype=torch.long).to(device)

    return X_tensor, y_tensor, len(encoder.classes_)

def run_mass_gdv_analysis():
    print(f"Inizio analisi GDV su {len(UCI_DATASETS)} dataset...\n")
    
    risultati = []
    
    for nome, uci_id in UCI_DATASETS.items():
        print(f"Elaborazione: {nome} (ID: {uci_id})...", end=" ")
        start_time = time.time()
        
        try:
            dataset = fetch_ucirepo(id=uci_id)
            X_raw = dataset.data.features
            y_raw = dataset.data.targets
            
            if y_raw is None or y_raw.empty:
                raise ValueError("Nessun target (y) trovato nel dataset.")
                
            X_tensor, y_tensor, num_classes = clean_and_prepare_data(X_raw, y_raw)
            
            gdv = calculate_gdv(X_tensor, y_tensor)
            
            tempo = time.time() - start_time
            print(f"Completato! GDV: {gdv:.4f} | Tempo: {tempo:.2f}s")
            
            risultati.append({
                "Dataset": nome,
                "UCI_ID": uci_id,
                "Num_Campioni": X_tensor.shape[0],
                "Num_Feature_Post_Clean": X_tensor.shape[1],
                "Num_Classi": num_classes,
                "GDV_Spazio_Input": round(gdv, 4),
                "Status": "OK",
                "Note_Errore": ""
            })
            
        except Exception as e:
            messaggio_errore = f"{type(e).__name__}: {str(e)}"
            print(f"FALLITO! Motivo: {messaggio_errore}")
            
            risultati.append({
                "Dataset": nome,
                "UCI_ID": uci_id,
                "Num_Campioni": None,
                "Num_Feature_Post_Clean": None,
                "Num_Classi": None,
                "GDV_Spazio_Input": None,
                "Status": "Fallito",
                "Note_Errore": messaggio_errore
            })
            
    df_report = pd.DataFrame(risultati)
    csv_filename = "report_gdv_uci_dettagliato.csv"
    df_report.to_csv(csv_filename, index=False)
    
    print("\n" + "="*60)
    print("ANALISI COMPLETATA!")
    print(f"I risultati sono stati salvati nel file: '{csv_filename}'")
    print("="*60)
    
    return df_report


if __name__ == "__main__":
    df_finale = run_mass_gdv_analysis()
    
    print("\n--- RIEPILOGO ESECUZIONE ---")
    totali = len(df_finale)
    successi = len(df_finale[df_finale['Status'] == 'OK'])
    fallimenti = totali - successi
    print(f"Dataset analizzati con successo: {successi}/{totali}")
    print(f"Dataset falliti: {fallimenti}/{totali}")
    
    if fallimenti > 0:
        print("\n--- DETTAGLIO FALLIMENTI ---")
        df_falliti = df_finale[df_finale['Status'] == 'Fallito']
        print(df_falliti[['Dataset', 'Note_Errore']].to_string(index=False))